In [4]:
import os
import pandas as pd
import pymol
from pymol import cmd

pymol.finish_launching(['pymol', '-cq'])

def render_docking_pose(receptor_path, ligand_path, output_image):
    cmd.reinitialize()
    cmd.load(receptor_path, "receptor")
    cmd.load(ligand_path, "ligand")
    cmd.hide("all")
    cmd.show("cartoon", "receptor")
    cmd.color("gray70", "receptor")
    cmd.show("sticks", "ligand")
    cmd.util.cbc("ligand")
    cmd.select("zinc", "resn ZN")
    cmd.show("spheres", "zinc")
    cmd.color("purple", "zinc")
    cmd.select("pocket", "receptor within 5 of ligand")
    cmd.show("lines", "pocket")
    cmd.color("cyan", "pocket")
    cmd.distance("hbonds", "ligand", "pocket", mode=2)
    cmd.hide("labels", "hbonds")
    cmd.color("yellow", "hbonds")
    cmd.bg_color("white")
    
    # Close up
    cmd.zoom("pocket", buffer=2.0)
    cmd.png(output_image.replace(".png", "_closeup.png"), 
            width=1920, height=1080, dpi=300, ray=1)
    
    # Wide shot
    cmd.zoom("receptor", buffer=5.0)
    cmd.png(output_image.replace(".png", "_wide.png"), 
            width=1920, height=1080, dpi=300, ray=1)
    
    print(f"✅ Rendered: {output_image}")

results_df = pd.read_csv("results_df.csv")
top_hits = results_df[results_df['score'] <= -9.0]

print(f"Found {len(top_hits)} candidates. Starting rendering...")

output_dir = "renders"
os.makedirs(output_dir, exist_ok=True)

receptor_file = "receptor/1GKC_clean.pdb"

for index, row in top_hits.iterrows():
    ligand_id = row['compound_name']
    ligand_file = f"ligands/{ligand_id}_out.pdbqt"
    output_file = os.path.join(output_dir, f"{ligand_id}_interaction.png")
    
    if os.path.exists(ligand_file):
        render_docking_pose(receptor_file, ligand_file, output_file)
    else:
        print(f"⚠️ Could not find {ligand_file}")



print("🎉 All renders complete!")

Found 6 candidates. Starting rendering...
✅ Rendered: renders\ligand_0_interaction.png
✅ Rendered: renders\ligand_12_interaction.png
✅ Rendered: renders\ligand_13_interaction.png
✅ Rendered: renders\ligand_16_interaction.png
✅ Rendered: renders\ligand_3_interaction.png
✅ Rendered: renders\ligand_1_interaction.png
🎉 All renders complete!
